In [1]:
import os
import gc
from glob import glob
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import polars as pl

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.base import BaseEstimator, ClassifierMixin

import lightgbm as lgb

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
Path='c:\\Users\\eugen\\Downloads\\home-credit-credit-risk-model-stability\\csv_files\\'

In [2]:
class VotingModel(BaseEstimator, ClassifierMixin):
    def __init__(self, estimators):
        super().__init__()
        self.estimators = estimators
        
    def fit(self, X, y=None):
        return self
    
    def predict(self, X):
        y_preds = [estimator.predict(X) for estimator in self.estimators]
        return np.mean(y_preds, axis=0)
    
    def predict_proba(self, X):
        y_preds = [estimator.predict_proba(X) for estimator in self.estimators]
        return np.mean(y_preds, axis=0)

In [4]:
class Pipeline:
    @staticmethod
    def set_table_dtypes(df):
        for col in df.columns:
            if col in ["case_id", "WEEK_NUM", "num_group1", "num_group2"]:
                df = df.with_columns(pl.col(col).cast(pl.Int32))
            elif col in ["date_decision"]:
                df = df.with_columns(pl.col(col).cast(pl.Date))
            elif col[-1] in ("P", "A"):
                df = df.with_columns(pl.col(col).cast(pl.Float64))
            elif col[-1] in ("M",):
                df = df.with_columns(pl.col(col).cast(pl.String))
            elif col[-1] in ("D",):
                df = df.with_columns(pl.col(col).cast(pl.Date))            

        return df
    
    @staticmethod
    def handle_dates(df):
        for col in df.columns:
            if col[-1] in ("D",):
                df = df.with_columns(pl.col(col) - pl.col("date_decision"))
                df = df.with_columns(pl.col(col).dt.total_days())
                df = df.with_columns(pl.col(col).cast(pl.Float32))
                
        df = df.drop("date_decision", "MONTH")

        return df
    
    @staticmethod
    def filter_cols(df):
        for col in df.columns:
            if col not in ["target", "case_id", "WEEK_NUM"]:
                isnull = df[col].is_null().mean()

                if isnull > 0.95:
                    df = df.drop(col)

        for col in df.columns:
            if (col not in ["target", "case_id", "WEEK_NUM"]) & (df[col].dtype == pl.String):
                freq = df[col].n_unique()

                if (freq == 1) | (freq > 200):
                    df = df.drop(col)

        return df

In [5]:
class Aggregator:
    @staticmethod
    def num_expr(df):
        cols = [col for col in df.columns if col[-1] in ("P", "A")]

        expr_max = [pl.max(col).alias(f"max_{col}") for col in cols]

        return expr_max

    @staticmethod
    def date_expr(df):
        cols = [col for col in df.columns if col[-1] in ("D",)]

        expr_max = [pl.max(col).alias(f"max_{col}") for col in cols]

        return expr_max

    @staticmethod
    def str_expr(df):
        cols = [col for col in df.columns if col[-1] in ("M",)]
        
        expr_max = [pl.max(col).alias(f"max_{col}") for col in cols]

        return expr_max

    @staticmethod
    def other_expr(df):
        cols = [col for col in df.columns if col[-1] in ("T", "L")]
        
        expr_max = [pl.max(col).alias(f"max_{col}") for col in cols]

        return expr_max
    
    @staticmethod
    def count_expr(df):
        cols = [col for col in df.columns if "num_group" in col]

        expr_max = [pl.max(col).alias(f"max_{col}") for col in cols]

        return expr_max

    @staticmethod
    def get_exprs(df):
        exprs = Aggregator.num_expr(df) + \
                Aggregator.date_expr(df) + \
                Aggregator.str_expr(df) + \
                Aggregator.other_expr(df) + \
                Aggregator.count_expr(df)

        return exprs

In [6]:
def read_file(path, depth=None):
    df = pl.read_parquet(path)
    df = df.pipe(Pipeline.set_table_dtypes)
    
    if depth in [1, 2]:
        df = df.group_by("case_id").agg(Aggregator.get_exprs(df))
    
    return df

def read_files(regex_path, depth=None):
    chunks = []
    for path in glob(str(regex_path)):
        df = pl.read_parquet(path)
        df = df.pipe(Pipeline.set_table_dtypes)
        
        if depth in [1, 2]:
            df = df.group_by("case_id").agg(Aggregator.get_exprs(df))
        
        chunks.append(df)
        
    df = pl.concat(chunks, how="vertical_relaxed")
    df = df.unique(subset=["case_id"])
    
    return df

In [7]:
def feature_eng(df_base, depth_0, depth_1, depth_2):
    df_base = (
        df_base
        .with_columns(
            month_decision = pl.col("date_decision").dt.month(),
            weekday_decision = pl.col("date_decision").dt.weekday(),
        )
    )
        
    for i, df in enumerate(depth_0 + depth_1 + depth_2):
        df_base = df_base.join(df, how="left", on="case_id", suffix=f"_{i}")
        
    df_base = df_base.pipe(Pipeline.handle_dates)
    
    return df_base

In [8]:
def to_pandas(df_data, cat_cols=None):
    df_data = df_data.to_pandas()
    
    if cat_cols is None:
        cat_cols = list(df_data.select_dtypes("object").columns)
    
    df_data[cat_cols] = df_data[cat_cols].astype("category")
    
    return df_data, cat_cols

In [9]:
import os
os.chdir('c:\\Users\\eugen\\Downloads\\home-credit-credit-risk-model-stability\\')

In [10]:
from pathlib import Path
ROOT = Path('c:/Users/eugen/Downloads/home-credit-credit-risk-model-stability/')
TRAIN_DIR = ROOT / "parquet_files" / "train"
TEST_DIR = ROOT / "parquet_files" / "test"

In [11]:
data_store = {
    "df_base": read_file(TRAIN_DIR / "train_base.parquet"),
    "depth_0": [
        read_file(TRAIN_DIR / "train_static_cb_0.parquet"),
        read_files(TRAIN_DIR / "train_static_0_*.parquet"),
    ],
    "depth_1": [
        read_files(TRAIN_DIR / "train_applprev_1_*.parquet", 1),
        read_file(TRAIN_DIR / "train_tax_registry_a_1.parquet", 1),
        read_file(TRAIN_DIR / "train_tax_registry_b_1.parquet", 1),
        read_file(TRAIN_DIR / "train_tax_registry_c_1.parquet", 1),
        read_files(TRAIN_DIR / "train_credit_bureau_a_1_*.parquet", 1),
        read_file(TRAIN_DIR / "train_credit_bureau_b_1.parquet", 1),
        read_file(TRAIN_DIR / "train_other_1.parquet", 1),
        read_file(TRAIN_DIR / "train_person_1.parquet", 1),
        read_file(TRAIN_DIR / "train_deposit_1.parquet", 1),
        read_file(TRAIN_DIR / "train_debitcard_1.parquet", 1),
    ],
    "depth_2": [
        read_file(TRAIN_DIR / "train_credit_bureau_b_2.parquet", 2),
        read_files(TRAIN_DIR / "train_credit_bureau_a_2_*.parquet", 2),
    ]
}

In [12]:
df_train = feature_eng(**data_store)

print("train data shape:\t", df_train.shape)

train data shape:	 (1526659, 472)


In [13]:
data_store = {
    "df_base": read_file(TEST_DIR / "test_base.parquet"),
    "depth_0": [
        read_file(TEST_DIR / "test_static_cb_0.parquet"),
        read_files(TEST_DIR / "test_static_0_*.parquet"),
    ],
    "depth_1": [
        read_files(TEST_DIR / "test_applprev_1_*.parquet", 1),
        read_file(TEST_DIR / "test_tax_registry_a_1.parquet", 1),
        read_file(TEST_DIR / "test_tax_registry_b_1.parquet", 1),
        read_file(TEST_DIR / "test_tax_registry_c_1.parquet", 1),
        read_files(TEST_DIR / "test_credit_bureau_a_1_*.parquet", 1),
        read_file(TEST_DIR / "test_credit_bureau_b_1.parquet", 1),
        read_file(TEST_DIR / "test_other_1.parquet", 1),
        read_file(TEST_DIR / "test_person_1.parquet", 1),
        read_file(TEST_DIR / "test_deposit_1.parquet", 1),
        read_file(TEST_DIR / "test_debitcard_1.parquet", 1),
    ],
    "depth_2": [
        read_file(TEST_DIR / "test_credit_bureau_b_2.parquet", 2),
        read_files(TEST_DIR / "test_credit_bureau_a_2_*.parquet", 2),
    ]
}

In [14]:
df_test = feature_eng(**data_store)

print("test data shape:\t", df_test.shape)

test data shape:	 (10, 471)


In [15]:
df_train = df_train.pipe(Pipeline.filter_cols)
df_test = df_test.select([col for col in df_train.columns if col != "target"])

print("train data shape:\t", df_train.shape)
print("test data shape:\t", df_test.shape)

train data shape:	 (1526659, 361)
test data shape:	 (10, 360)


In [16]:
df_train, cat_cols = to_pandas(df_train)
df_test, cat_cols = to_pandas(df_test, cat_cols)

In [19]:
from sklearn.model_selection import train_test_split
import lightgbm as lgb
import lightgbm.callback as lgb


X = df_train.drop(columns=["target", "case_id", "WEEK_NUM"])
y = df_train["target"]
weeks = df_train["WEEK_NUM"]

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

params = {
    "boosting_type": "gbdt",
    "objective": "binary",
    "metric": "auc",
    "max_depth": 8,
    "learning_rate": 0.05,
    "n_estimators": 1000,
    "colsample_bytree": 0.8, 
    "colsample_bynode": 0.8,
    "verbose": -1,
    "random_state": 42
    #"device": "gpu",
}


model = lgb.LGBMClassifier(**params)
  
model.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    callbacks=[lgb.log_evaluation(100), lgb.early_stopping(100)]
)


Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.838618
[200]	valid_0's auc: 0.848081
[300]	valid_0's auc: 0.851495
[400]	valid_0's auc: 0.853188
[500]	valid_0's auc: 0.854122
[600]	valid_0's auc: 0.854549
[700]	valid_0's auc: 0.854918
[800]	valid_0's auc: 0.855279
[900]	valid_0's auc: 0.855462
[1000]	valid_0's auc: 0.855764
Did not meet early stopping. Best iteration is:
[998]	valid_0's auc: 0.85577


LGBMClassifier(colsample_bynode=0.8, colsample_bytree=0.8, learning_rate=0.05,
               max_depth=8, metric='auc', n_estimators=1000, objective='binary',
               random_state=42, verbose=-1)

In [27]:
X = df_train.drop(columns=["target", "case_id", "WEEK_NUM"])
y = df_train["target"]
weeks = df_train["WEEK_NUM"]

cv = StratifiedGroupKFold(n_splits=5, shuffle=False)

params = {
    "boosting_type": "gbdt",
    "objective": "binary",
    "metric": "auc",
    "max_depth": 8,
    "learning_rate": 0.05,
    "n_estimators": 1000,
    "colsample_bytree": 0.8, 
    "colsample_bynode": 0.8,
    "verbose": -1,
    "random_state": 42,
    "device": "gpu",
}

fitted_models = []

for idx_train, idx_valid in cv.split(X, y, groups=weeks):
    X_train, y_train = X.iloc[idx_train], y.iloc[idx_train]
    X_valid, y_valid = X.iloc[idx_valid], y.iloc[idx_valid]

    model = lgb.LGBMClassifier(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_valid, y_valid)],
        callbacks=[lgb.log_evaluation(100), lgb.early_stopping(100)]
    )

    fitted_models.append(model)

model = VotingModel(fitted_models)

Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.837898
[200]	valid_0's auc: 0.84731
[300]	valid_0's auc: 0.850356
[400]	valid_0's auc: 0.851829
[500]	valid_0's auc: 0.852286
[600]	valid_0's auc: 0.852682
[700]	valid_0's auc: 0.852958
[800]	valid_0's auc: 0.853175
[900]	valid_0's auc: 0.853467
[1000]	valid_0's auc: 0.853535
Did not meet early stopping. Best iteration is:
[995]	valid_0's auc: 0.853553
Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.838104
[200]	valid_0's auc: 0.847232
[300]	valid_0's auc: 0.850138


X_test = df_test.drop(columns=["WEEK_NUM"])
X_test = X_test.set_index("case_id")

y_pred = pd.Series(model.predict_proba(X_test)[:, 1], index=X_test.index)

In [ ]:
df_subm = pd.read_csv(ROOT / "sample_submission.csv")
df_subm = df_subm.set_index("case_id")

df_subm["score"] = y_pred

In [ ]:
df_subm.to_csv("submission.csv")

In [ ]:
train_applprev_1_0= pl.read_csv('train\\train_applprev_1_0.csv')
train_applprev_1_1= pl.read_csv('train\\train_applprev_1_1.csv')
train_applprev_2= pl.read_csv('train\\train_applprev_2.csv')

train_applprev=pl.concat([train_applprev_1_0,train_applprev_1_1,train_applprev_2],axis=0)


DuplicateError: unable to hstack, column with name "num_group1_right" already exists

In [7]:
#combine all train_credit_breaue files
tcba10  =  pl.read_csv("train\\train_credit_bureau_a_1_0.csv")
tcba11  =  pl.read_csv("train\\train_credit_bureau_a_1_1.csv")
tcba12  =  pl.read_csv("train\\train_credit_bureau_a_1_2.csv")
tcba13  =  pl.read_csv("train\\train_credit_bureau_a_1_3.csv")




tcba20  =  pl.read_csv("train\\train_credit_bureau_a_2_0.csv")
tcba21  =  pl.read_csv("train\\train_credit_bureau_a_2_1.csv")
tcba22  =  pl.read_csv("train\\train_credit_bureau_a_2_2.csv")
tcba23  =  pl.read_csv("train\\train_credit_bureau_a_2_3.csv")
tcba24  =  pl.read_csv("train\\train_credit_bureau_a_2_4.csv")
tcba25  =  pl.read_csv("train\\train_credit_bureau_a_2_5.csv")
tcba26  =  pl.read_csv("train\\train_credit_bureau_a_2_6.csv")
tcba27  =  pl.read_csv("train\\train_credit_bureau_a_2_7.csv")
tcba28  =  pl.read_csv("train\\train_credit_bureau_a_2_8.csv")
tcba29  =  pl.read_csv("train\\train_credit_bureau_a_2_9.csv")
tcba210  = pl.read_csv("train\\train_credit_bureau_a_2_10.csv")



dataframes = [
    tcba10, tcba11, tcba12, tcba13,
    tcba20, tcba21, tcba22, tcba23, tcba24,
    tcba25, tcba26, tcba27, tcba28, tcba29, tcba210
]

# Concatenate all DataFrames in the list
combined_df = pl.concat(dataframes)



In [25]:
def set_table_dtypes(df: pl.DataFrame) -> pl.DataFrame:
    float_cols = [col for col in df.columns if col[-1] in ("P", "A")]
    return df.with_columns(pl.col(float_cols).cast(pl.Float64))

def convert_strings(df: pd.DataFrame) -> pd.DataFrame:
    string_cols = [col for col in df.columns if df[col].dtype.name in ['object', 'string']]
    for col in string_cols:
        df[col] = df[col].astype("string").astype('category')
        df[col] = df[col].cat.add_categories(["Unknown"]).fillna("Unknown")
    return df

def read_and_process_csv(file_path: str) -> pl.DataFrame:
    return pl.read_csv(file_path).pipe(set_table_dtypes)

def read_and_concat_csv(file_paths: list[str]) -> pl.DataFrame:
    return pl.concat([read_and_process_csv(file_path) for file_path in file_paths], how="vertical_relaxed")

# Train data
train_basetable = pl.read_csv(dataPath + "csv_files/train/train_base.csv")
train_static = read_and_concat_csv([
    dataPath + "csv_files/train/train_static_0_0.csv",
    dataPath + "csv_files/train/train_static_0_1.csv"
])
train_static_cb = read_and_process_csv(dataPath + "csv_files/train/train_static_cb_0.csv")
train_person_1 = read_and_process_csv(dataPath + "csv_files/train/train_person_1.csv")
train_credit_bureau_b_2 = read_and_process_csv(dataPath + "csv_files/train/train_credit_bureau_b_2.csv")

# Test data
test_basetable = pl.read_csv(dataPath + "csv_files/test/test_base.csv")
test_static = read_and_concat_csv([
    dataPath + "csv_files/test/test_static_0_0.csv",
    dataPath + "csv_files/test/test_static_0_1.csv",
    dataPath + "csv_files/test/test_static_0_2.csv"
])
test_static_cb = read_and_process_csv(dataPath + "csv_files/test/test_static_cb_0.csv")
test_person_1 = read_and_process_csv(dataPath + "csv_files/test/test_person_1.csv")
test_credit_bureau_b_2 = read_and_process_csv(dataPath + "csv_files/test/test_credit_bureau_b_2.csv")

In [23]:
def set_table_dtypes(df: pl.DataFrame) -> pl.DataFrame:
    # implement here all desired dtypes for tables
    # the following is just an example
    for col in df.columns:
        # last letter of column name will help you determine the type
        if col[-1] in ("P", "A"):
            df = df.with_columns(pl.col(col).cast(pl.Float64).alias(col))

    return df

def convert_strings(df: pd.DataFrame) -> pd.DataFrame:
    for col in df.columns:  
        if df[col].dtype.name in ['object', 'string']:
            df[col] = df[col].astype("string").astype('category')
            current_categories = df[col].cat.categories
            new_categories = current_categories.to_list() + ["Unknown"]
            new_dtype = pd.CategoricalDtype(categories=new_categories, ordered=True)
            df[col] = df[col].astype(new_dtype)
    return df

# Train data
train_basetable = pl.read_csv(dataPath + "csv_files/train/train_base.csv")
train_static = read_and_concat_csv([
    dataPath + "csv_files/train/train_static_0_0.csv",
    dataPath + "csv_files/train/train_static_0_1.csv"
])
train_static_cb = read_and_process_csv(dataPath + "csv_files/train/train_static_cb_0.csv")
train_person_1 = read_and_process_csv(dataPath + "csv_files/train/train_person_1.csv")
train_credit_bureau_b_2 = read_and_process_csv(dataPath + "csv_files/train/train_credit_bureau_b_2.csv")

# Test data
test_basetable = pl.read_csv(dataPath + "csv_files/test/test_base.csv")
test_static = read_and_concat_csv([
    dataPath + "csv_files/test/test_static_0_0.csv",
    dataPath + "csv_files/test/test_static_0_1.csv",
    dataPath + "csv_files/test/test_static_0_2.csv"
])
test_static_cb = read_and_process_csv(dataPath + "csv_files/test/test_static_cb_0.csv")
test_person_1 = read_and_process_csv(dataPath + "csv_files/test/test_person_1.csv")
test_credit_bureau_b_2 = read_and_process_csv(dataPath + "csv_files/test/test_credit_bureau_b_2.csv")

In [22]:



train_basetable = pl.read_csv(dataPath + "csv_files/train/train_base.csv")
train_static = pl.concat(
    [
        pl.read_csv(dataPath + "csv_files/train/train_static_0_0.csv").pipe(set_table_dtypes),
        pl.read_csv(dataPath + "csv_files/train/train_static_0_1.csv").pipe(set_table_dtypes),
    ],
    how="vertical_relaxed",
)
train_static_cb = pl.read_csv(dataPath + "csv_files/train/train_static_cb_0.csv").pipe(set_table_dtypes)
train_person_1 = pl.read_csv(dataPath + "csv_files/train/train_person_1.csv").pipe(set_table_dtypes) 
train_credit_bureau_b_2 = pl.read_csv(dataPath + "csv_files/train/train_credit_bureau_b_2.csv").pipe(set_table_dtypes) 


test_basetable = pl.read_csv(dataPath + "csv_files/test/test_base.csv")
test_static = pl.concat(
    [
        pl.read_csv(dataPath + "csv_files/test/test_static_0_0.csv").pipe(set_table_dtypes),
        pl.read_csv(dataPath + "csv_files/test/test_static_0_1.csv").pipe(set_table_dtypes),
        pl.read_csv(dataPath + "csv_files/test/test_static_0_2.csv").pipe(set_table_dtypes),
    ],
    how="vertical_relaxed",
)
test_static_cb = pl.read_csv(dataPath + "csv_files/test/test_static_cb_0.csv").pipe(set_table_dtypes)
test_person_1 = pl.read_csv(dataPath + "csv_files/test/test_person_1.csv").pipe(set_table_dtypes) 
test_credit_bureau_b_2 = pl.read_csv(dataPath + "csv_files/test/test_credit_bureau_b_2.csv").pipe(set_table_dtypes) 


AttributeError: 'DataFrame' object has no attribute 'with_dtypes'

In [26]:
# We need to use aggregation functions in tables with depth > 1, so tables that contain num_group1 column or 
# also num_group2 column.
train_person_1_feats_1 = train_person_1.group_by("case_id").agg(
    pl.col("mainoccupationinc_384A").max().alias("mainoccupationinc_384A_max"),
    (pl.col("incometype_1044T") == "SELFEMPLOYED").max().alias("mainoccupationinc_384A_any_selfemployed")
)

# Here num_group1=0 has special meaning, it is the person who applied for the loan.
train_person_1_feats_2 = train_person_1.select(["case_id", "num_group1", "housetype_905L"]).filter(
    pl.col("num_group1") == 0
).drop("num_group1").rename({"housetype_905L": "person_housetype"})

# Here we have num_goup1 and num_group2, so we need to aggregate again.
train_credit_bureau_b_2_feats = train_credit_bureau_b_2.group_by("case_id").agg(
    pl.col("pmts_pmtsoverdue_635A").max().alias("pmts_pmtsoverdue_635A_max"),
    (pl.col("pmts_dpdvalue_108P") > 31).max().alias("pmts_dpdvalue_108P_over31")
)

# We will process in this examples only A-type and M-type columns, so we need to select them.
selected_static_cols = []
for col in train_static.columns:
    if col[-1] in ("A", "M"):
        selected_static_cols.append(col)
print(selected_static_cols)

selected_static_cb_cols = []
for col in train_static_cb.columns:
    if col[-1] in ("A", "M"):
        selected_static_cb_cols.append(col)
print(selected_static_cb_cols)

# Join all tables together.
data = train_basetable.join(
    train_static.select(["case_id"]+selected_static_cols), how="left", on="case_id"
).join(
    train_static_cb.select(["case_id"]+selected_static_cb_cols), how="left", on="case_id"
).join(
    train_person_1_feats_1, how="left", on="case_id"
).join(
    train_person_1_feats_2, how="left", on="case_id"
).join(
    train_credit_bureau_b_2_feats, how="left", on="case_id"
)

['amtinstpaidbefduel24m_4187115A', 'annuity_780A', 'annuitynextmonth_57A', 'avginstallast24m_3658937A', 'avglnamtstart24m_4525187A', 'avgoutstandbalancel6m_4187114A', 'avgpmtlast12m_4525200A', 'credamount_770A', 'currdebt_22A', 'currdebtcredtyperange_828A', 'disbursedcredamount_1113A', 'downpmt_116A', 'inittransactionamount_650A', 'lastapprcommoditycat_1041M', 'lastapprcommoditytypec_5251766M', 'lastapprcredamount_781A', 'lastcancelreason_561M', 'lastotherinc_902A', 'lastotherlnsexpense_631A', 'lastrejectcommoditycat_161M', 'lastrejectcommodtypec_5251769M', 'lastrejectcredamount_222A', 'lastrejectreason_759M', 'lastrejectreasonclient_4145040M', 'maininc_215A', 'maxannuity_159A', 'maxannuity_4075009A', 'maxdebt4_972A', 'maxinstallast24m_3658928A', 'maxlnamtstart6m_4525199A', 'maxoutstandbalancel12m_4187113A', 'maxpmtlast3m_4525190A', 'previouscontdistrict_112M', 'price_1097A', 'sumoutstandtotal_3546847A', 'sumoutstandtotalest_4493215A', 'totaldebt_9A', 'totalsettled_863A', 'totinstallas

In [7]:
test_person_1_feats_1 = test_person_1.group_by("case_id").agg(
    pl.col("mainoccupationinc_384A").max().alias("mainoccupationinc_384A_max"),
    (pl.col("incometype_1044T") == "SELFEMPLOYED").max().alias("mainoccupationinc_384A_any_selfemployed")
)

test_person_1_feats_2 = test_person_1.select(["case_id", "num_group1", "housetype_905L"]).filter(
    pl.col("num_group1") == 0
).drop("num_group1").rename({"housetype_905L": "person_housetype"})

test_credit_bureau_b_2_feats = test_credit_bureau_b_2.group_by("case_id").agg(
    pl.col("pmts_pmtsoverdue_635A").max().alias("pmts_pmtsoverdue_635A_max"),
    (pl.col("pmts_dpdvalue_108P") > 31).max().alias("pmts_dpdvalue_108P_over31")
)

data_submission = test_basetable.join(
    test_static.select(["case_id"]+selected_static_cols), how="left", on="case_id"
).join(
    test_static_cb.select(["case_id"]+selected_static_cb_cols), how="left", on="case_id"
).join(
    test_person_1_feats_1, how="left", on="case_id"
).join(
    test_person_1_feats_2, how="left", on="case_id"
).join(
    test_credit_bureau_b_2_feats, how="left", on="case_id"
)

In [9]:
from sklearn.model_selection import train_test_split
# Convert Polars Series of case IDs to a Python list
case_ids = data["case_id"].unique().to_list()
case_ids_train, case_ids_temp = train_test_split(case_ids, train_size=0.6, random_state=1)
case_ids_valid, case_ids_test = train_test_split(case_ids_temp, train_size=0.5, random_state=1)

# Columns prediction logic
cols_pred = [col for col in data.columns if col[-1].isupper() and col[:-1].islower()]

def from_polars_to_pandas(case_ids):
    # Adjust filtering to work with a list of case_ids
    filtered_data = data.filter(pl.col("case_id").is_in(case_ids))
    return (
        filtered_data.select(["case_id", "WEEK_NUM", "target"]).to_pandas(),
        filtered_data.select(cols_pred).to_pandas(),
        filtered_data.select("target").to_pandas()
    )

# Convert data for each split
base_train, X_train, y_train = from_polars_to_pandas(case_ids_train)
base_valid, X_valid, y_valid = from_polars_to_pandas(case_ids_valid)
base_test, X_test, y_test = from_polars_to_pandas(case_ids_test)

# Assuming convert_strings is defined elsewhere
# Apply string conversion function (if needed)
for df in [X_train, X_valid, X_test]:
    df = convert_strings(df)

In [10]:
print(f"Train: {X_train.shape}")
print(f"Valid: {X_valid.shape}")
print(f"Test: {X_test.shape}")

Train: (915995, 48)
Valid: (305332, 48)
Test: (305332, 48)


In [12]:
import lightgbm as lgb

lgb_train = lgb.Dataset(X_train, label=y_train)
lgb_valid = lgb.Dataset(X_valid, label=y_valid, reference=lgb_train)

params = {
    "boosting_type": "gbdt",
    "objective": "binary",
    "metric": "auc",
    "max_depth": 3,
    "num_leaves": 31,
    "learning_rate": 0.05,
    "feature_fraction": 0.9,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "n_estimators": 1000,
    "verbose": -1,
}

gbm = lgb.train(
    params,
    lgb_train,
    valid_sets=lgb_valid,
    callbacks=[lgb.log_evaluation(50), lgb.early_stopping(10)]
)

C:\Users\eugen\AppData\Roaming\Python\Python312\site-packages\lightgbm\engine.py:172: UserWarning: Found `n_estimators` in params. Will use it instead of argument
  _log_warning(f"Found `{alias}` in params. Will use it instead of argument")


Training until validation scores don't improve for 10 rounds
[50]	valid_0's auc: 0.703682
[100]	valid_0's auc: 0.720802
[150]	valid_0's auc: 0.728145
[200]	valid_0's auc: 0.733023
[250]	valid_0's auc: 0.736343
[300]	valid_0's auc: 0.739195
[350]	valid_0's auc: 0.74136
[400]	valid_0's auc: 0.743087
[450]	valid_0's auc: 0.744576
[500]	valid_0's auc: 0.74582
[550]	valid_0's auc: 0.74689
[600]	valid_0's auc: 0.747841
[650]	valid_0's auc: 0.748727
[700]	valid_0's auc: 0.749547
[750]	valid_0's auc: 0.750433
[800]	valid_0's auc: 0.751548
[850]	valid_0's auc: 0.752265
[900]	valid_0's auc: 0.752663
[950]	valid_0's auc: 0.753171
[1000]	valid_0's auc: 0.753564
Did not meet early stopping. Best iteration is:
[992]	valid_0's auc: 0.753572


In [13]:
for base, X in [(base_train, X_train), (base_valid, X_valid), (base_test, X_test)]:
    y_pred = gbm.predict(X, num_iteration=gbm.best_iteration)
    base["score"] = y_pred

print(f'The AUC score on the train set is: {roc_auc_score(base_train["target"], base_train["score"])}') 
print(f'The AUC score on the valid set is: {roc_auc_score(base_valid["target"], base_valid["score"])}') 
print(f'The AUC score on the test set is: {roc_auc_score(base_test["target"], base_test["score"])}')  

NameError: name 'roc_auc_score' is not defined

In [14]:
X_submission = data_submission[cols_pred].to_pandas()
X_submission = convert_strings(X_submission)
categorical_cols = X_train.select_dtypes(include=['category']).columns

for col in categorical_cols:
    train_categories = set(X_train[col].cat.categories)
    submission_categories = set(X_submission[col].cat.categories)
    new_categories = submission_categories - train_categories
    X_submission.loc[X_submission[col].isin(new_categories), col] = "Unknown"
    new_dtype = pd.CategoricalDtype(categories=train_categories, ordered=True)
    X_train[col] = X_train[col].astype(new_dtype)
    X_submission[col] = X_submission[col].astype(new_dtype)

y_submission_pred = gbm.predict(X_submission, num_iteration=gbm.best_iteration)